# Laboratorios de Pandas: limpieza, análisis y combinación de datos (Instacart)

**Actividad:** GA1-220501094-01-AA1-EV06 · Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Programa:** Análisis y Desarrollo de Software · SENA · Ficha 3223873

**Aprendiz:** Nicolás Muñoz Hernández

**Fecha:** 11 de noviembre de 2026

**Avance de este cuaderno:** Lección 2

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

| Parte | Contenido |
|---|---|
| Preparación | Carga de los cinco archivos CSV |
| Lección 2 | Limpieza y estandarización · Laboratorio guiado 2 · Reto 2 |
| Lección 3 | Filtrado, ordenamiento y agrupación · Laboratorio guiado 3 · Reto 3 |
| Lección 4 | Combinación de datasets · Laboratorio guiado 4 · Reto 4 |

## Preparación: carga de los datos

Los cinco archivos están en `data/raw/` (se conservan sin modificar) y usan `;` como separador. Los resultados procesados se guardarán en `data/processed/`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# El cuaderno vive en notebooks/, así que los datos están un nivel arriba.
RUTA = next((c for c in [Path('../data/raw'), Path('data/raw')] if (c / 'instacart_orders.csv').exists()), None)
assert RUTA is not None, 'No encuentro los CSV: ponlos en data/raw/ (ver README.md).'
RAIZ = RUTA.parent.parent
SALIDA = RAIZ / 'data' / 'processed'
INFORMES = RAIZ / 'reports'
SALIDA.mkdir(parents=True, exist_ok=True)
INFORMES.mkdir(parents=True, exist_ok=True)

fmt = lambda n: f'{n:,.0f}'.replace(',', '.')      # 1234567 -> 1.234.567

try:
    import pyarrow
    print('pyarrow', pyarrow.__version__)
except ImportError:
    print('AVISO: falta pyarrow. Instálalo con  python -m pip install pyarrow  y reinicia el kernel.')
print('pandas', pd.__version__)
print('Datos leídos desde:', RUTA.resolve())

pyarrow 26.0.1
pandas 3.1.0
Datos leídos desde: C:\Users\IK\Documents\instacart-analisis\data\raw


In [2]:
orders = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')

for nombre, df in {'orders': orders, 'products': products, 'aisles': aisles,
                   'departments': departments, 'order_prods': order_prods}.items():
    print(f'{nombre:12s}', df.shape)
# Deberías ver: (478967, 6) (49694, 4) (134, 2) (21, 2) (4545007, 4)

orders       (478967, 6)
products     (49694, 4)
aisles       (134, 2)
departments  (21, 2)
order_prods  (4545007, 4)


---
# Lección 2 · Limpieza y estandarización de datos crudos

Para cada problema hay tres opciones: **eliminar, corregir o conservar**, y la decisión depende de qué significa el dato. Se sigue siempre este orden: duplicados → tipos → nulos → texto → duplicados implícitos → verificación.

## 2.1 Ejemplos de la lección

### Duplicados exactos: `duplicated` y `drop_duplicates`

In [3]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)
# Deberías ver 30 filas: 15 pedidos y sus 15 copias, todas en order_dow = 3 y hora 2

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Todas las copias ocurren el mismo día y a la misma hora (miércoles, 2 a. m.), lo que apunta a un fallo del sistema y no a pedidos reales. **Decisión: eliminar.**

In [4]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)
# Deberías ver (478952, 6) True

(478952, 6) True


### Tipos de dato con `astype`

Se reducen los enteros al tamaño necesario y se usa `Int16` (con mayúscula, *nullable*) donde hay nulos.

In [5]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')
# Deberías ver Antes: 23.0 MB y Después: 7.2 MB (si la ejecutas dos veces, "Antes" ya muestra la tabla optimizada)

Antes: 23.0 MB
Después: 7.2 MB


### Nulos: comprobar la causa antes de decidir

In [6]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())
# Deberías ver True: cada nulo es el primer pedido de un cliente

True


Cada nulo de `days_since_prior_order` es un **primer pedido** (no existe pedido anterior). **Decisión: conservar el nulo.** Rellenar con 0 inventaría que el cliente volvió el mismo día y bajaría la media de espera.

In [7]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())
# Deberías ver una sola combinación (100, 21) con 1258 productos, y luego 0 nulos

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


Los productos sin nombre pertenecen todos al pasillo y departamento `missing`. **Decisión: corregir** con un valor explícito (`'Unknown'`); el producto existe y no se puede borrar.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [8]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())
# Deberías ver Espacios dobles: 403, Espacio invisible: 16 y, tras limpiar, 0 0

Espacios dobles: 403
Espacio invisible: 16
0 0


### Duplicados implícitos

Aparecen solo después de estandarizar (por ejemplo `American Cheese slices` y `American Cheese Slices`).

In [9]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))
# Deberías ver 141 y "268 de 280"

141
268 de 280


Los nombres repetidos tienen `product_id` distintos y casi todos aparecen en pedidos. **Decisión: conservar y documentar.** Borrarlos dejaría pedidos apuntando a productos inexistentes; en los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [10]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)
# Deberías ver (478952, 6): aplicada a una tabla ya limpia, no cambia nada

(478952, 6)


In [11]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada: todas las comprobaciones pasaron')

Limpieza verificada: todas las comprobaciones pasaron


## 2.2 Laboratorio guiado 2 · Pipeline de limpieza con informe de cambios

Se parte de los archivos **crudos** y se encadenan funciones pequeñas con `pipe`; después de cada paso se anota el estado de la tabla para poder responder "¿qué le hiciste a los datos?" con cifras.

### Paso 1: recargar los datos crudos

In [12]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)
# Deberías ver (478967, 6) (49694, 4) (4545007, 4)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [13]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [14]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [15]:
registro = []   # se reinicia aquí para no duplicar filas si se vuelve a ejecutar

orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [16]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


In [17]:
ob = informe.set_index(['tabla', 'paso'])
dup_quitados = ob.loc[('orders', '0 crudo'), 'filas'] - ob.loc[('orders', '1 sin duplicados'), 'filas']
nulos_quitados = ob.loc[('orders', '0 crudo'), 'nulos'] - ob.loc[('orders', '1 sin duplicados'), 'nulos']
mem_o0, mem_o2 = ob.loc[('orders', '0 crudo'), 'memoria_mb'], ob.loc[('orders', '2 tipos'), 'memoria_mb']
mem_p0, mem_p1 = ob.loc[('order_prods', '0 crudo'), 'memoria_mb'], ob.loc[('order_prods', '1 tipos'), 'memoria_mb']
nulos_op = ob.loc[('order_prods', '1 tipos'), 'nulos']
nulos_pr0, nulos_pr1 = ob.loc[('products', '0 crudo'), 'nulos'], ob.loc[('products', '1 nombres'), 'nulos']
display(Markdown(
    f"**Interpretación del informe:**\n\n"
    f"- Quitar duplicados eliminó {dup_quitados} filas de `orders` y {nulos_quitados} nulos (copias de pedidos que eran primeros pedidos).\n"
    f"- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de {mem_o0} a {mem_o2} MB y la de `order_prods` de {mem_p0} a {mem_p1} MB.\n"
    f"- Limpiar nombres pasó los nulos de `products` de {nulos_pr0} a {nulos_pr1}.\n"
    f"- Los {nulos_op} nulos de `order_prods` (`add_to_cart_order`) siguen ahí a propósito: se decidió conservarlos."
))

**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas de `orders` y 2 nulos (copias de pedidos que eran primeros pedidos).
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23.0 a 7.2 MB y la de `order_prods` de 145.4 a 54.5 MB.
- Limpiar nombres pasó los nulos de `products` de 1258 a 0.
- Los 836 nulos de `order_prods` (`add_to_cart_order`) siguen ahí a propósito: se decidió conservarlos.

### Paso 6: validar el resultado

In [18]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))
# Deberías ver "Validación superada" y "True True"

Validación superada
True True


El pipeline produce exactamente las mismas tablas que la limpieza manual: la limpieza queda **reproducible** y validada con `assert`.

## 2.3 Reto 2 · Siete ejercicios

### Ejercicio 1 · Duplicados exactos, pares repetidos y valores de `reordered` en `order_prods`

In [19]:
dup_exactos = int(order_prods.duplicated().sum())
dup_pares = int(order_prods.duplicated(subset=['order_id', 'product_id']).sum())
valores_reordered = sorted(order_prods['reordered'].unique().tolist())
print('Filas duplicadas exactas:', dup_exactos)
print('Pares order_id + product_id repetidos:', dup_pares)
print('Valores de reordered:', valores_reordered)

limpio = dup_exactos == 0 and dup_pares == 0 and set(valores_reordered) <= {0, 1}
display(Markdown(
    f"**Interpretación:** hay {fmt(dup_exactos)} duplicados exactos y {fmt(dup_pares)} pares repetidos; `reordered` solo contiene {valores_reordered}. "
    + ("La tabla no necesita limpieza en estos tres puntos." if limpio else "Hay que revisar estos registros antes de analizar.")
))

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Interpretación:** hay 0 duplicados exactos y 0 pares repetidos; `reordered` solo contiene [0, 1]. La tabla no necesita limpieza en estos tres puntos.

### Ejercicio 2 · Nombres duplicados o con espacios en `aisles` y `departments`

In [20]:
resultados = []
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    resultados.append({
        'tabla': nombre,
        'nombres duplicados': int(df[col].duplicated().sum()),
        'espacios en los extremos': int((df[col] != df[col].str.strip()).sum()),
        'espacios dobles': int(df[col].str.contains(r'\s{2,}', regex=True).sum()),
        'espacio invisible': int(df[col].str.contains('\xa0', regex=False).sum()),
    })
res2 = pd.DataFrame(resultados)
display(res2)
necesita = res2.drop(columns='tabla').sum().sum() > 0
display(Markdown("**Interpretación:** " + ("hay problemas de texto que conviene estandarizar." if necesita else "no hay duplicados ni espacios sobrantes, así que `aisles` y `departments` **no necesitan limpieza de texto**.")))

,tabla,nombres duplicados,espacios en los extremos,espacios dobles,espacio invisible
0,aisles,0,0,0,0
1,departments,0,0,0,0


**Interpretación:** no hay duplicados ni espacios sobrantes, así que `aisles` y `departments` **no necesitan limpieza de texto**.

### Ejercicio 3 · Los nulos de `add_to_cart_order`

In [21]:
sin_posicion = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_con_nulos = sin_posicion['order_id'].unique()
nulos_por_pedido = sin_posicion.groupby('order_id').size()
tamano_pedidos = order_prods[order_prods['order_id'].isin(pedidos_con_nulos)].groupby('order_id').size()
max_posicion = int(order_prods['add_to_cart_order'].max())

print('Nulos:', len(sin_posicion))
print('Pedidos afectados:', len(pedidos_con_nulos))
print('Productos mínimos en esos pedidos:', int(tamano_pedidos.min()), '| máximo:', int(tamano_pedidos.max()))
print('Máxima posición registrada en toda la tabla:', max_posicion)
print('Nulos por pedido (mín, máx):', int(nulos_por_pedido.min()), int(nulos_por_pedido.max()))

if int(tamano_pedidos.min()) > max_posicion:
    causa = (f"todos los pedidos afectados tienen más de {max_posicion} productos y la posición máxima registrada es {max_posicion}: "
             f"el sistema dejó de numerar los productos después de la posición {max_posicion}. Es un **límite del registro**, no un error aleatorio.")
else:
    causa = "los pedidos afectados no se explican solo por su tamaño; habría que investigar otra causa con el área de datos."
display(Markdown(f"**Interpretación:** {len(sin_posicion)} nulos en {len(pedidos_con_nulos)} pedidos; cada uno tiene como mínimo {int(tamano_pedidos.min())} productos. Explicación: {causa}"))

Nulos: 836
Pedidos afectados: 70
Productos mínimos en esos pedidos: 65 | máximo: 127
Máxima posición registrada en toda la tabla: 64
Nulos por pedido (mín, máx): 1 63


**Interpretación:** 836 nulos en 70 pedidos; cada uno tiene como mínimo 65 productos. Explicación: todos los pedidos afectados tienen más de 64 productos y la posición máxima registrada es 64: el sistema dejó de numerar los productos después de la posición 64. Es un **límite del registro**, no un error aleatorio.

### Ejercicio 4 · Cuántos nombres cambiaron con la estandarización

In [22]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
tienen_nombre = products_crudo['product_name'].notna()
cambiaron = tienen_nombre & (products_crudo['product_name'] != products['product_name'])
print('Nombres no nulos:', int(tienen_nombre.sum()))
print('Nombres que cambiaron:', int(cambiaron.sum()))
ejemplos = pd.DataFrame({'antes': products_crudo.loc[cambiaron, 'product_name'],
                         'después': products.loc[cambiaron, 'product_name']}).head(5)
display(ejemplos)
display(Markdown(f"**Interpretación:** {fmt(cambiaron.sum())} de {fmt(tienen_nombre.sum())} nombres ({cambiaron.sum() / tienen_nombre.sum() * 100:.1f} %) cambiaron: espacios dobles, espacios invisibles y mayúsculas distintas. Sin este paso, el mismo producto podría contarse como varios."))

Nombres no nulos: 48436
Nombres que cambiaron: 6321


,antes,después
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Interpretación:** 6.321 de 48.436 nombres (13.1 %) cambiaron: espacios dobles, espacios invisibles y mayúsculas distintas. Sin este paso, el mismo producto podría contarse como varios.

### Ejercicio 5 · Día de la semana como categoría ordenada

In [23]:
nombres_dia = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles', 4: 'jueves', 5: 'viernes', 6: 'sábado'}
orden_dias = list(nombres_dia.values())

dia_texto = orders['order_dow'].map(nombres_dia)
dia_cat = pd.Series(pd.Categorical(dia_texto, categories=orden_dias, ordered=True), index=orders.index)

mem_texto = dia_texto.memory_usage(deep=True)
mem_cat = dia_cat.memory_usage(deep=True)
print('Memoria como texto:', round(mem_texto / 1e6, 2), 'MB')
print('Memoria como categoría:', round(mem_cat / 1e6, 2), 'MB')
print('Primer y último día del orden:', dia_cat.min(), '→', dia_cat.max())
print(dia_cat.value_counts().sort_index())
display(Markdown(f"**Interpretación:** la categoría ocupa {mem_cat / mem_texto * 100:.0f} % de la memoria del texto y además tiene un **orden** real (de domingo a sábado), por lo que `sort_values` y los gráficos respetan la semana y no el alfabeto."))

Memoria como texto: 7.14 MB
Memoria como categoría: 0.48 MB
Primer y último día del orden: domingo → sábado
domingo      84090
lunes        82185
martes       65833
miércoles    60897
jueves       59810
viernes      63488
sábado       62649
Name: count, dtype: int64


**Interpretación:** la categoría ocupa 7 % de la memoria del texto y además tiene un **orden** real (de domingo a sábado), por lo que `sort_values` y los gráficos respetan la semana y no el alfabeto.

### Ejercicio 6 · Filas de `order_prods` con productos `'Unknown'`

In [24]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
mascara = order_prods['product_id'].isin(ids_unknown)
filas_unknown = int(mascara.sum())
pedidos_unknown = order_prods.loc[mascara, 'order_id'].nunique()
print('Productos Unknown en el catálogo:', len(ids_unknown))
print('Filas de order_prods con productos Unknown:', filas_unknown)
print('Pedidos en que aparecen:', pedidos_unknown)
display(Markdown(
    f"**Interpretación:** {fmt(filas_unknown)} artículos ({filas_unknown / len(order_prods) * 100:.2f} % de `order_prods`) en {fmt(pedidos_unknown)} pedidos corresponden a productos sin nombre. "
    f"**No los eliminaría:** son compras reales; borrarlas distorsionaría el tamaño de los pedidos y las tasas de recompra. "
    f"Se conservan y, cuando se analicen productos por nombre, se excluyen solo en ese análisis."
))

Productos Unknown en el catálogo: 1258
Filas de order_prods con productos Unknown: 9547
Pedidos en que aparecen: 8254


**Interpretación:** 9.547 artículos (0.21 % de `order_prods`) en 8.254 pedidos corresponden a productos sin nombre. **No los eliminaría:** son compras reales; borrarlas distorsionaría el tamaño de los pedidos y las tasas de recompra. Se conservan y, cuando se analicen productos por nombre, se excluyen solo en ese análisis.

### Ejercicio 7 · Reemplazar `'missing'` por un nulo real en una copia de `departments`

In [25]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.tail(3))

con_original = products.merge(departments, on='department_id', how='left').groupby('department').size()
con_copia = products.merge(departments_copia, on='department_id', how='left').groupby('department').size()
print('Productos contados por departamento (original):', fmt(con_original.sum()))
print('Productos contados por departamento (copia con NA):', fmt(con_copia.sum()))
display(Markdown(
    f"**Interpretación:** con el nulo real, `groupby` descarta en silencio {fmt(con_original.sum() - con_copia.sum())} productos porque ignora las claves nulas por defecto. "
    f"En la tabla original conviene mantener `'missing'` como una categoría visible para que esos artículos sigan apareciendo en las agrupaciones."
))

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Productos contados por departamento (original): 49.694
Productos contados por departamento (copia con NA): 48.436


**Interpretación:** con el nulo real, `groupby` descarta en silencio 1.258 productos porque ignora las claves nulas por defecto. En la tabla original conviene mantener `'missing'` como una categoría visible para que esos artículos sigan apareciendo en las agrupaciones.

---
## Verificación del avance (Lección 2)

Antes de guardar: **Restart** y luego **Run All**. Todas las celdas deben ejecutarse sin error y las comprobaciones `assert` no deben mostrar mensajes de fallo.